# **Layer choice and the faithfulness of a map: LayerCAM, HiResCAM, Grad-CAM++**

A practice for the module ["Localization and the CAM family"](https://open-xai-platform.web.app).

The lesson makes three claims, and none of them has to be taken on trust — each can be checked
with a number:

1. Grad-CAM is poor on early layers because averaging the gradient over positions cancels it;
2. for HiResCAM the sum of the map equals the class logit up to the bias;
3. the pointing game measures agreement with human expectation, not the faithfulness of the model.

That is what we will do. It runs on a CPU in under a minute.

In [ ]:
import io
import urllib.request

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'

model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V2).eval()

tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


img = tf(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB')).unsqueeze(0)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]

LAYERS = {'layer2': model.layer2, 'layer3': model.layer3, 'layer4': model.layer4}

logits = model(img)
predicted = int(logits.argmax())
print(f'predicted class: {predicted} {names[predicted]}')

## 1. The tools

The three methods differ by one line each — and that is the whole point of the lesson.
Grad-CAM averages the gradient over positions and gets **one weight per channel**. LayerCAM
does not average at all: **every element** has its own weight. HiResCAM multiplies the gradient
by the activation **element-wise**.

Note the `relu=False` argument of `hires_cam`: it will be needed in section 3, and not by
accident.

In [ ]:
def capture(layer, x, cls):
    """Activations of the layer and the gradient of the class logit — one forward and one backward pass."""
    store = {}
    handle = layer.register_forward_hook(lambda m, i, o: store.__setitem__('a', o))
    out = model(x)
    handle.remove()
    A = store['a']
    A.retain_grad()
    model.zero_grad()
    out[0, cls].backward()
    return A.detach(), A.grad.detach(), out.detach()


def grad_cam(A, G):
    """Grad-CAM: one weight per channel, the gradient averaged over all positions."""
    alpha = G.mean(dim=(2, 3), keepdim=True)
    return F.relu((alpha * A).sum(1))[0]


def layer_cam(A, G):
    """LayerCAM: a weight for every element of the map, no averaging."""
    return F.relu((F.relu(G) * A).sum(1))[0]


def hires_cam(A, G, relu=True):
    """HiResCAM: element-wise product of the gradient and the activation."""
    m = (G * A).sum(1)[0]
    return F.relu(m) if relu else m


def upscale(cam):
    return F.interpolate(cam[None, None], (224, 224), mode='bilinear',
                         align_corners=False)[0, 0]

## 2. Why Grad-CAM is poor on early layers

The lesson explains this by cancelling: inside a single channel the gradient is positive in
some places and negative in others, and averaging destroys them against each other.

This can be checked directly. For every channel we compute two quantities: the magnitude of
the averaged gradient and the averaged magnitude of the gradient. Their ratio is the measure
of cancelling:

- **close to zero** — the gradients inside the channel cancel almost completely;
- **one** — there is no cancelling at all, the sign is the same across the channel.

In [ ]:
print(f'layer    size      channels    cancelling   near-zero weights')
for name, layer in LAYERS.items():
    A, G, _ = capture(layer, img, predicted)
    averaged = G.mean(dim=(2, 3))[0].abs()      # magnitude of the averaged gradient of the channel
    magnitude = G.abs().mean(dim=(2, 3))[0]     # averaged magnitude of the gradient of the channel
    ratio = (averaged / (magnitude + 1e-12)).mean().item()
    dead = (averaged < 0.01 * averaged.max()).float().mean().item()
    print(f'{name:9}{str(tuple(A.shape[-2:])):10}{A.shape[1]:<10}{ratio:12.3f}{dead*100:19.1f}%')

**Look at the `layer4` row.** The ratio is **exactly one**, and this is neither
an accident nor rounding. After the last convolutional layer of a ResNet come a global average
pooling and one linear layer, so

$$y^c = \sum_k w^c_k \frac{1}{Z}\sum_{ij} A^k_{ij} + b
\qquad\Longrightarrow\qquad
\frac{\partial y^c}{\partial A^k_{ij}} = \frac{w^c_k}{Z}$$

The gradient **does not depend on the position** — it is the same across the whole map of the
channel. There is nothing to average and nothing to cancel. That is why "the last convolutional
layer" is not a convention for Grad-CAM but the only layer where its averaging loses nothing.

On `layer2` and `layer3` there is no such equality, the ratio falls — and the spatial
information goes with it. That is exactly what LayerCAM cures.

**Task 1.** Build the maps of both methods on all three layers and look at them. The cell below
does that. A question for you: on which layer is the difference between Grad-CAM and LayerCAM
most visible, and does it agree with what the cancelling ratio showed?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for col, (name, layer) in enumerate(LAYERS.items()):
    A, G, _ = capture(layer, img, predicted)
    for row, (title, cam) in enumerate((('Grad-CAM', grad_cam(A, G)),
                                        ('LayerCAM', layer_cam(A, G)))):
        axes[row, col].imshow(upscale(cam).numpy(), cmap='jet')
        axes[row, col].set_title(f'{title} · {name}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 3. The HiResCAM guarantee: checking the equality

The lesson claims: if only a global average pooling and one linear layer follow the chosen
layer, then **the sum of all elements of the HiResCAM map equals the class logit exactly**, up
to the bias. Not approximately — exactly.

A claim like that is either checked or not worth the paper. Let us check it.

In [ ]:
A, G, out = capture(model.layer4, img, predicted)
bias = model.fc.bias[predicted].item()
logit = out[0, predicted].item()

for label, relu in ((f'without ReLU', False), (f'with ReLU', True)):
    total = hires_cam(A, G, relu=relu).sum().item()
    print(f'{label:12} {total:9.4f} + {bias:7.4f} = {total + bias:9.4f}   '
          f'logit {logit:.4f}   discrepancy {abs(total + bias - logit):.2e}')

**A discrepancy of order $10^{-7}$ is zero at floating-point precision.**
The equality holds.

Now look at the second row. **With ReLU the equality breaks.** And that is no small matter: the
formula of the method in the lesson is written with a ReLU, while the guarantee is true for the
map **before** it. ReLU cuts off the negative contributions — exactly the ones that speak
against the class — and the sum stops matching the logit.

So the precise statement is this: HiResCAM **decomposes** the logit if the map is taken without
the ReLU. The ReLU is there to make the map visible to the eye, and it is the very thing that
spoils the property the method was taken for. Keep that in mind when you write "the sum of the
map equals the logit" in a report.

Let us make sure it is not about a lucky class.

In [ ]:
weak = int(logits[0].argsort()[-200])
A, G, out = capture(model.layer4, img, weak)
bias, logit = model.fc.bias[weak].item(), out[0, weak].item()

print(f'a weak class: {weak} {names[weak]}')
for label, relu in ((f'without ReLU', False), (f'with ReLU', True)):
    total = hires_cam(A, G, relu=relu).sum().item()
    print(f'{label:12} discrepancy {abs(total + bias - logit):.3e}')

**Task 2.** For a weak class the discrepancy with ReLU is an order of magnitude
larger. Explain why: what happens to the negative contributions when the model is not confident
in the class?

Check your guess — compute what share of the elements of the map is negative for the strong
class and for the weak one.

In [ ]:
# Your code here

## 4. The pointing game and its main flaw

The metric is simple: take the peak of the map and see whether it landed inside the box of the
object. There is a cat and a dog in the picture, and the model confidently sees the dog — let
us see where the map points.

In [ ]:
# The boxes are drawn by eye on the 224 by 224 picture — this is the "human annotation" the lesson talks about
BOXES = {'cat': (10, 40, 110, 200), 'dog': (115, 30, 215, 205)}

A, G, _ = capture(model.layer4, img, predicted)
peak = int(upscale(grad_cam(A, G)).argmax())
py, px = divmod(peak, 224)
print(f'the peak of the map is at ({px}, {py})')
for label, (x0, y0, x1, y1) in BOXES.items():
    hit = x0 <= px <= x1 and y0 <= py <= y1
    print(f'  {label:8} ({x0}, {y0})–({x1}, {y1})  hit' if hit
          else f'  {label:8} ({x0}, {y0})–({x1}, {y1})  miss')

The peak landed inside the dog box and outside the cat box. For the dog class that
is "correct", for the cat class it is an "error".

**And now the main thing.** Imagine that the model recognizes the dog partly by the sofa it is
lying on — and that this is an honest regularity of its training data. The map will point at
the sofa, the pointing game will count a miss, and we will declare the method bad. But neither
the method nor the model was wrong: we were, by assuming that a correct explanation has to
point at the object.

That is why the pointing game is a quick filter and not a criterion. It compares the map with
**our expectation**, while insertion and deletion ask the model itself.

**Task 3.** Build the map for the cat class and compute the pointing game for both boxes. It
will turn out that for the cat the metric counts a hit and for the dog a miss. The same method,
the same model, different answers — depending on which class we asked about. What does that say
about the applicability of the metric to comparing **methods** with each other?

In [ ]:
# Your code here

## 5. Grad-CAM++ next to Grad-CAM

Grad-CAM++ re-weights the positions by curvature: a point where a growth of the activation
quickly raises the class score gets a larger weight. The lesson promises that this stops a small
object from drowning in the averaging.

Let us look at the numbers: compare how concentrated the two maps are.

In [ ]:
def grad_cam_pp(A, G):
    """Grad-CAM++: the weight of a position through curvature. The exp(logit) factor cancels and is not needed."""
    g2, g3 = G ** 2, G ** 3
    denom = 2 * g2 + A.sum(dim=(2, 3), keepdim=True) * g3
    alpha = torch.where(denom != 0, g2 / denom, torch.zeros_like(denom))
    weights = (alpha * F.relu(G)).sum(dim=(2, 3), keepdim=True)
    return F.relu((weights * A).sum(1))[0]


def concentration(cam):
    """Share of the map mass in the top 10 % of elements: the higher, the more concentrated the map."""
    values = cam.flatten()
    k = max(1, values.numel() // 10)
    return (torch.topk(values, k).values.sum() / (values.sum() + 1e-9)).item()


A, G, out = capture(model.layer4, img, predicted)
for title, cam in (('Grad-CAM', grad_cam(A, G)), ('Grad-CAM++', grad_cam_pp(A, G))):
    peak = int(upscale(cam).argmax())
    py, px = divmod(peak, 224)
    print(f'{title:11} peak ({px:3}, {py:3})   share of mass in the top 10 % {concentration(cam):.3f}')

**Task 4.** The Grad-CAM++ map is less concentrated — the mass is spread wider.
That is expected: the method pulls out precisely the weak responses that Grad-CAM damps by
averaging.

Something to think about: does "less concentrated" mean "worse"? Check it not by eye: compute
the deletion AUC for both maps (the code is in the `HW8` notebook) and compare the numbers.
If Grad-CAM++ is more spread out but its deletion is lower — which of the two will you take to
the client?

In [ ]:
# Your code here

## What to take away from this notebook

- **"The last convolutional layer" is not a convention for Grad-CAM.** It is the only layer
  where the gradient does not depend on the position, which means averaging loses nothing. On
  early layers the cancelling ratio falls, and the spatial information goes with it.
- **The HiResCAM guarantee is real — and fragile.** The sum of the map equals the logit to the
  seventh decimal, but only without the ReLU. With the ReLU the equality breaks, and the more
  so the less confident the model is.
- **The pointing game answers the wrong question.** It measures the agreement of the map with
  our annotation. The same method gets both "correct" and "error" depending on which class was
  asked about.
- **The difference between methods is measurable.** Not a single conclusion in this notebook
  was reached by looking at a picture, and that is the only way of choosing a method that can
  be trusted.